In [2]:
import time
import requests
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)

s = requests.Session()
s.headers["User-Agent"] = "petro-tracker research notebook"

def get(url):
    time.sleep(0.15)
    r = s.get(url, timeout=30)
    r.raise_for_status()
    return r.json()

In [3]:
raw = get("https://www.sec.gov/files/company_tickers.json")
print(type(raw), len(raw))
list(raw.items())[:3]

<class 'dict'> 10444


[('0', {'cik_str': 1045810, 'ticker': 'NVDA', 'title': 'NVIDIA CORP'}),
 ('1', {'cik_str': 320193, 'ticker': 'AAPL', 'title': 'Apple Inc.'}),
 ('2', {'cik_str': 1652044, 'ticker': 'GOOGL', 'title': 'Alphabet Inc.'})]

In [10]:
tick = pd.DataFrame(raw.values())
tick["cik10"] = tick["cik_str"].astype(str).str.zfill(10)
print(tick.shape)
print(tick["ticker"].is_unique, tick["cik_str"].is_unique)
tick.head()

(10444, 4)
True False


,cik_str,ticker,title,cik10
0,1045810,NVDA,NVIDIA CORP,0001045810
1,320193,AAPL,Apple Inc.,0000320193
2,1652044,GOOGL,Alphabet Inc.,0001652044
3,789019,MSFT,MICROSOFT CORP,0000789019
4,1018724,AMZN,AMAZON COM INC,0001018724


In [11]:
dup = tick[tick["cik_str"].duplicated(keep=False)].sort_values("cik_str")
print(len(dup))
dup.head(8)

3857


,cik_str,ticker,title,cik10
191,2809,AEM,AGNICO EAGLE MINES LTD,0000002809
9732,2809,AEMRF,AGNICO EAGLE MINES LTD,0000002809
7644,3124,AKZOF,AKZO NOBEL NV,0000003124
896,3124,AKZOY,AKZO NOBEL NV,0000003124
907,4457,UHAL,U-Haul Holding Co /NV/,0000004457
7665,4457,UHAL-B,U-Haul Holding Co /NV/,0000004457
7729,5513,UNMA,Unum Group,0000005513
744,5513,UNM,Unum Group,0000005513


In [12]:
peers = ["XOM", "CVX", "COP", "OXY", "VLO", "MPC", "PSX", "DOW", "LYB", "EOG"]
mine = tick[tick["ticker"].isin(peers)].set_index("ticker").loc[peers]
mine[["cik10", "title"]]

,cik10,title
ticker,,
XOM,0002115436,ExxonMobil Holdings Corp
CVX,0000093410,CHEVRON CORP
COP,0001163165,CONOCOPHILLIPS
OXY,0000797468,OCCIDENTAL PETROLEUM CORP /DE/
VLO,0001035002,VALERO ENERGY CORP/TX
MPC,0001510295,Marathon Petroleum Corp
PSX,0001534701,Phillips 66
DOW,0001751788,DOW INC.
LYB,0001489393,LyondellBasell Industries N.V.


In [13]:
tick[tick["title"].str.contains("exxon", case=False)]

,cik_str,ticker,title,cik10
15,2115436,XOM,ExxonMobil Holdings Corp,0002115436
5153,2014337,NPT,Texxon Holding Ltd,0002014337


In [14]:
h = get("https://data.sec.gov/submissions/CIK0002115436.json")
print(h["name"], h["sic"], h["tickers"], h["exchanges"])
print(h["formerNames"])
print(len(h["filings"]["recent"]["form"]), "filings")
old = get("https://data.sec.gov/submissions/CIK0000034088.json")
print(old["name"], old["tickers"], old["exchanges"])

ExxonMobil Holdings Corp 2911 ['XOM'] ['NYSE']
[]
34 filings
EXXON MOBIL CORP [] []


In [15]:
mine.loc["XOM", "cik10"] = "0000034088"
mine.loc["XOM", "cik_str"] = 34088
mine[["cik10", "title"]].head(3)

,cik10,title
ticker,,
XOM,0000034088,ExxonMobil Holdings Corp
CVX,0000093410,CHEVRON CORP
COP,0001163165,CONOCOPHILLIPS


In [16]:
xom = get("https://data.sec.gov/submissions/CIK0000034088.json")
for k, v in xom.items():
    if k != "filings":
        print(f"{k:35s} {str(v)[:70]}")

cik                                 0000034088
entityType                          operating
sic                                 2911
sicDescription                      Petroleum Refining
ownerOrg                            01 Energy & Transportation
insiderTransactionForOwnerExists    1
insiderTransactionForIssuerExists   1
name                                EXXON MOBIL CORP
tickers                             []
exchanges                           []
ein                                 135409005
lei                                 None
description                         
website                             
investorWebsite                     
category                            Large accelerated filer
fiscalYearEnd                       1231
stateOfIncorporation                NJ
stateOfIncorporationDescription     NJ
addresses                           {'mailing': {'street1': '22777 SPRINGWOODS VILLAGE PARKWAY', 'street2'
phone                               9729406000
flags     

In [17]:
print(xom["sic"], xom["sicDescription"])
print(xom["tickers"], xom["exchanges"])
xom["addresses"]["business"]

2911 Petroleum Refining
[] []


{'street1': '22777 SPRINGWOODS VILLAGE PARKWAY',
 'street2': None,
 'city': 'SPRING',
 'stateOrCountry': 'TX',
 'zipCode': '77389-1425',
 'stateOrCountryDescription': 'TX',
 'isForeignLocation': None,
 'foreignStateTerritory': None,
 'country': None,
 'countryCode': None}

In [18]:
f = xom["filings"]
print(list(f["recent"].keys()))
print(len(f["recent"]["accessionNumber"]), "rows in recent")
print(f["files"])

['accessionNumber', 'filingDate', 'reportDate', 'acceptanceDateTime', 'act', 'form', 'fileNumber', 'filmNumber', 'items', 'core_type', 'size', 'isXBRL', 'isInlineXBRL', 'isXBRLNumeric', 'primaryDocument', 'primaryDocDescription']
1001 rows in recent
[{'name': 'CIK0000034088-submissions-001.json', 'filingCount': 2002, 'filingFrom': '2005-10-04', 'filingTo': '2020-01-02'}, {'name': 'CIK0000034088-submissions-002.json', 'filingCount': 554, 'filingFrom': '1994-03-04', 'filingTo': '2005-10-03'}]


In [19]:
rec = pd.DataFrame(f["recent"])
rec["filingDate"] = pd.to_datetime(rec["filingDate"])
print(rec["filingDate"].min(), rec["filingDate"].max())
rec.head()

2020-01-03 00:00:00 2026-09-25 00:00:00


,accessionNumber,filingDate,reportDate,acceptanceDateTime,act,form,fileNumber,filmNumber,items,core_type,size,isXBRL,isInlineXBRL,isXBRLNumeric,primaryDocument,primaryDocDescription
0,0001193125-26-401391,2026-09-25,,2026-09-25T01:10:19.000Z,33,424B2,333-293558,261406038,,424B2,671354,1,1,0.0,d137820d424b2.htm,424B2
1,0001193125-26-399637,2026-09-23,,2026-09-23T21:17:57.000Z,34,FWP,333-293558,261401690,,FWP,33470,0,0,0.0,d188276dfwp.htm,FWP
2,0001193125-26-396148,2026-09-21,,2026-09-21T13:07:48.000Z,33,424B3,333-293558,261392401,,424B3,529180,0,0,0.0,d137820d424b3.htm,424B3
3,0000093751-26-000461,2026-08-07,,2026-08-07T13:32:39.000Z,34,SCHEDULE 13G,005-81094,261251841,,SCHEDULE 13G,8521,0,0,0.0,xslSCHEDULE_13G_X02/primary_doc.xml,
4,0000034088-26-000093,2026-08-03,2026-06-30,2026-08-03T18:55:38.000Z,34,10-Q,001-02256,261232241,,XBRL,14847031,1,1,1.0,xom-20260630.htm,FORM 10-Q


In [20]:
rec["form"].value_counts().head(15)

form
4           300
DEFA14A     260
8-K         107
DFAN14A      68
PX14A6G      45
425          33
3            27
10-Q         20
SC 13G/A     19
424B2        15
SD            9
UPLOAD        9
FWP           8
11-K          7
10-K          7
Name: count, dtype: int64

In [21]:
older = [get("https://data.sec.gov/submissions/" + x["name"]) for x in f["files"]]
older = pd.concat([pd.DataFrame(o) for o in older])
older["filingDate"] = pd.to_datetime(older["filingDate"])
allf = pd.concat([rec, older], ignore_index=True)
print(len(allf), allf["filingDate"].min().date(), allf["filingDate"].max().date())
allf["accessionNumber"].is_unique

3557 1994-03-04 2026-09-25


True